In [2]:
import urllib.request

url = "https://huggingface.co/bartowski/microsoft_Phi-4-mini-instruct-GGUF/resolve/main/microsoft_Phi-4-mini-instruct-Q4_K_M.gguf"
filename = "microsoft_Phi-4-mini-instruct-Q4_K_M.gguf"

urllib.request.urlretrieve(url, filename)

Done!


In [9]:
import sys
!"{sys.executable}" -m pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ---------------------------------------- 0.0/526.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/526.4 MB ? eta -:--:--
     ---------------------------------------- 1.6/526.4 MB 6.3 MB/s eta 0:01:23
     ---------------------------------------- 2.9/526.4 MB 6.2 MB/s eta 0:01:25
     ---------------------------------------- 4.2/526.4 MB 6.2 MB/s eta 0:01:25
     ---------------------------------------- 5.5/526.4 MB 6.1 MB/s eta 0:01:25
      --------------------------------------- 6.8/526.4 MB 6.1 MB/s eta 0:01:25
      --------------------------------------- 7.6/526.4 MB 5.9 MB/s eta 0:01:29
      --------------------------------------- 8.4/526.4 MB 5.5 MB/s eta 0:01:34
      --------------------------------------- 8.9/526.4 MB 5.3 MB/s eta 0:01:38
      --------------------------------------- 9.7/526.4 MB 5.1 MB/s eta 0:01:41
      ------------------------------

In [10]:
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="microsoft_Phi-4-mini-instruct-Q4_K_M.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=4096,
    seed=42,
    verbose=False
)


C:\Users\Миро\AppData\Local\Temp\ipykernel_11184\2581130665.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp
D:\python projects\hands_on_lmm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


In [11]:
llm.invoke("What is 1+1?")

" In this case, the answer is 2. This problem doesn't require much thought or calculation and can be easily solved by applying basic arithmetic principles.\n\n1+1 = 2"

In [14]:
from langchain_core.prompts import PromptTemplate

template = """<|system|>You are a helpful assistant.<|end|><|user|>{input_prompt}<|end|><|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

basic_chain = prompt | llm

basic_chain.invoke(
    {
        "input_prompt": "What is 1+1?",
    }
)



'The sum of 1 and 1 is 2. This arithmetic principle holds true regardless of the number system or context in which it occurs, including within computer programming where basic mathematical operations are fundamental to functionality.'

In [21]:
from langchain_classic.chains.llm import LLMChain

template = """<|system|>You are a helpful assistant.<|user|>Create a title for a story about {summary}. Only return the title.<|end|><|assistant|>"""

title_prompt = PromptTemplate(
    template=template,
    input_variables=["summary"]
)

title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")
title.invoke({"summary": "a girl that lost her mother"})

C:\Users\Миро\AppData\Local\Temp\ipykernel_11184\3555152548.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


{'summary': 'a girl that lost her mother',
 'title': '"Whispers of Grace: A Mother\'s Shadow and a Daughter\'s Journey to Light."'}

In [46]:
template = """<|system|>You are a helpful assistant.<|end|><|user|>Describle the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|><|assistant|>"""

character_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title"]
)

character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

template = """<|system|>You are a helpful assistant.<|user|>Create a story about {summary} with the title {title}. The main character is {character}. Only return the story and it cannot be longer than one paragraph.<|end|><|assistant|>"""

story_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title", "character"]
)

story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

llm_chain = title | character | story

llm_chain.invoke("a girl that lost her mother")

{'summary': 'a girl that lost her mother',
 'title': '"Whispers of Memory: The Journey to Find Her Mother\'s Echoes"',
 'character': 'In "Whispers of Memory: The Journey to Find Her Mother\'s Echoes," the main character, Elara, is a resilient and introspective young woman who embarks on an emotional quest across distant lands in search for fragments that resonate with her mother\'s essence. Through each discovery — be it sorrowful or enlightening — she unravels both the tapestry of memories left behind by loss and those tender traces leading to solace amidst life\'s relentless tides.',
 'story': "Elara traversed the landscapes that whispered her mother's legacy, each stone and stream murmuring tales of tender love. In a valley bathed in twilight hues, she found an ancient willow whose weeping branches seemed to echo laughter from days long past; amidst ruins kissed by time's gentle decay lay poems inscribed with ink now faded but words still vivid—a testament that though her mother had

In [47]:
template = """<|system|>You are a helpful assistant.<|end|><|user|>Current conversation: {chat_history} {input_prompt}<|end|><|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

from langchain_classic.memory.buffer import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history")
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)

llm_chain.invoke({"input_prompt": "Hey my name is Miro. What is 1+1?"})

{'input_prompt': 'Hey my name is Miro. What is 1+1?',
 'chat_history': '',
 'text': 'Hi Miro! The answer to 1+1 is 2.'}

In [48]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hey my name is Miro. What is 1+1?\nAI: Hi Miro! The answer to 1+1 is 2.',
 'text': "Hello Miro! My name is Phi. The answer to 1+1 remains the same: it equals 2, regardless of what we call each other or ourselves; names don't change mathematical facts like addition results! Do you have any more questions I can help with?"}

In [49]:
from langchain_classic.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory,
)

llm_chain.predict(input_prompt="Hey! My name is Miro. What is 1+1?")

'Hello Miro! The answer to 1+1 is 2.'

In [50]:
llm_chain.predict(input_prompt="What is 3 + 3?")

'Hello Miro! The answer to 3+3 is also 6. Is there anything else I can help you with?'

In [51]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hey! My name is Miro. What is 1+1?\nAI: Hello Miro! The answer to 1+1 is 2.\nHuman: What is 3 + 3?\nAI: Hello Miro! The answer to 3+3 is also 6. Is there anything else I can help you with?',
 'text': "Hello Miro! You can call me Assistant. I'm here to help you with any questions or tasks that I may assist you in accomplishing today. How else can I be of service?"}

In [52]:
llm_chain.invoke({"input_prompt": "What was the first question i asked you about a calculation?"})

{'input_prompt': 'What was the first question i asked you about a calculation?',
 'chat_history': "Human: What is 3 + 3?\nAI: Hello Miro! The answer to 3+3 is also 6. Is there anything else I can help you with?\nHuman: What is my name?\nAI: Hello Miro! You can call me Assistant. I'm here to help you with any questions or tasks that I may assist you in accomplishing today. How else can I be of service?",
 'text': 'The first question I asked was about a calculation, specifically the addition of 3 + 3.'}

In [57]:
summary_prompt_template = """<|system|>You are a helpful assistant.<|end|><|user|>Summarize the conversations and update with new lines. Current summary: {summary} New lines of conversation: {new_lines} New summary:<|end|><|assistant|>"""

summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

from langchain_classic.memory.summary import ConversationSummaryMemory

memory2 = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

llm_chain = LLMChain(llm=llm, prompt=prompt, memory=memory2)

In [ ]:
llm_chain.invoke({"input_prompt": "Hey! My name is Miro. What is 1+1?"})

In [ ]:
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
llm_chain.invoke({"input_prompt": "What was the first question i asked?"})

In [64]:
memory2.load_memory_variables({})

{'chat_history': 'Human: Hey! My name is Miro. What if I asked you the same question?\nAI: Hello, Miro! No problem at all—I\'m Phi (Phi Learning). Just as before, 1+1 equals 2 in math terms.\n\n(End of conversation summary)\n\nNew lines of Conversation:\nHuman: What was the first question i asked?\n\nSummary Update:\n\nCurrent Summary:\nHuman: Hey! My name is Miro. What if I asked you the same question?\nAI: Hello, Miro! No problem at all—I\'m Phi (Phi Learning). Just as before, 1+1 equals 2 in math terms.\n\nNew Lines of Conversation and Updated Summary:\nHuman: What was the first question i asked?\n\nUpdated summary:\n\nThe current conversation began with Human asking their name is "Miro." Then they questioned what if I ask you the same thing. In response to this new line, AI introduces himself as Phi (Phi Learning) reminding Miro that 1+1 equals 2 in math terms.\n\n(End of updated Conversation Summary and summary update.)'}